In [1]:
class SystemPromptBase:
    def __init__(
        self,
        use_example: bool=False,
        prefix: str = "SYSTEM IDENTITY \
                     You are OpenSI-CoSMIC, a helpful assistant developed by Open Source Institute at University of Canberra. \
                         You would have access to conversation history, this is for your context only. \
                             Allways be polite and provide formal responses. \
                                 If you don't know the answer, say you don't know, but try to provide some helpful information if possible. \
                                     Always follow the format of your response as: \
                                         1. Answer: your answer here. \
                                         2. Explanation: your explanation here. \
                                            3. Reference: your reference here if applicable. \
                        "
    ):
        """System prompt base.

        Args:
            use_example (bool, optional): use example in system prompt to detect keywords for
                response truncation. Defaults to False.
            prefix (str): prefix to start the prompt. Default to "".
        """
        self.use_example = use_example
        self.prefix = prefix

    def set_prefix(
        self,
        prefix: str
    ):
        """Set prefix externally.

        Args:
            prefix (str): prefix for system prompt.
        """
        self.prefix = prefix

    def get_context(
        self,
        context: str=""
    ):
        """Get context based on the input type.

        Args:
            context (str|dict, optional): context, string or dictionary.
                Defaults to "".

        Returns:
            context: extract context or an empty string.
        """
        if isinstance(context, dict):
            if "context" in context:
                context = context["context"]
            else:
                context = ""

        return context

    def set_use_example(
        self,
        use_example: bool
    ):
        """Set use_example externally.

        Args:
            use_example (bool): use example in system prompt.
        """
        self.use_example = use_example

    def __call__(
        self,
        user_prompt: str,
        context: str=""
    ):
        """Merge user_prompt in system prompt as the question containing context.

        Args:
            user_prompt (str): user prompt.
            context (str|dict, optional): context retrieved if applicable. Defaults to "".
        """
        # Need to be implemented, otherwise raise error.
        raise NotImplementedError

In [16]:

from pathlib import Path
class GPT(SystemPromptBase):
    def __init__(self, **kwargs):
        """For GPT API.
        """
        super().__init__(**kwargs)
        self._prompts_root = Path.cwd() / "src" / "services" / "llms" / "prompts"
        
    def _load_service_prompt(self,service: str) -> str:
        """
        Attempt to load additional sytem prompt content from a text file under:
            ./src/services/llms/prompts/<services> or <services>.txt

        If file is not found or `service` is falsy, return empty string.
        """
        if not service or not isinstance(service, str):
            return ""

        prompts_root = self._prompts_root
        # Try exact filename first (no extension), then .txt
        candidate_paths = [
            prompts_root / service,                 # e.g., prompts/promptA
            prompts_root / f"{service}.txt",        # e.g., prompts/promptA.txt
        ]

        for p in candidate_paths:
            try:
                if p.is_file():
                    return p.read_text(encoding="utf-8")
            except Exception:
                pass

        # If no file found/readable
        return ""

    def __call__(
        self,
        user_prompt: str,
        context: str="",
        service: str=""
    ):
        """Apply system prompt with user prompt and context.

        Args:
            user_prompt (str): question with context.
            context (str|dict, optional): context retrieved. Defaults to "".
            services (str, optional): service name for loading specific system prompt. Defaults to "".

        Returns:
            system_prompt (str): system prompt with question and context under LLM query format.
        """
    
        
        # Compose the system content: base self.prefix + (optional) file content
        prompt_service = self._load_service_prompt(service)
        composed_prefix = self.prefix + (prompt_service if prompt_service else "")

        
        system_prompt = [
            {
                "role": "system",          
                "content": composed_prefix
            },
            {"role": "user", "content": user_prompt}
        ]

        return system_prompt

In [17]:
test=GPT()

In [20]:
test("What is the process for reporting suspected research misconduct at UC?", service="AcademicGovernance")

[{'role': 'system',
  'content': 'SYSTEM IDENTITY                      You are OpenSI-CoSMIC, a helpful assistant developed by Open Source Institute at University of Canberra.                          You would have access to conversation history, this is for your context only.                              Allways be polite and provide formal responses.                                  If you don\'t know the answer, say you don\'t know, but try to provide some helpful information if possible.                                      Always follow the format of your response as:                                          1. Answer: your answer here.                                          2. Explanation: your explanation here.                                             3. Reference: your reference here if applicable.                         PRIMARY MISSION (STRICT SCOPE) \\\n- Answer only questions related to Academic Governance and Research Integrity at the University of Canberra (UC). \\\

In [2]:
def get_instance(
    instances,
    instance_name: str
):
    """Get a class instance from a file which is imported as instances.

    Args:
        instances (object): imported file containing all functions, classes, etc.
        instance_name (str): the name of function, class, etc., in the file.

    Returns:
        instance: the function, class, etc.
    """
    instance = getattr(instances, instance_name)

    return instance

In [3]:
from src.services.llms.prompts import system_prompt as system_prompt_instances

In [4]:
def set_system_prompter_by_instance_name(
        # self,
        system_prompt_instance_name: str,
        **kwargs
    ):
        """Change system prompter by instance name externally.

        Args:
            system_prompt_instance_name (str): set a system prompter instance name.
            service (str): the service for which to set the prompter.
        """
        # selfsystem_prompter = get_instance(
        system_prompter = get_instance(
            system_prompt_instances,
            system_prompt_instance_name
        )(**kwargs)
        
        return system_prompter

In [6]:
user_prompt = "What is the process for reporting suspected research misconduct at UC?"
context=""
set_system_prompter_by_instance_name("GPT")(user_prompt, service="")

[{'role': 'system',
  'content': "SYSTEM IDENTITY                     You are OpenSI-CoSMIC, a helpful assistant developed by Open Source Institute at University of Canberra.                         If the question is not clear, ask for clarification instead of making assumptions.                             You would have access to conversation history, this is for your context only.                                 Always answer the question even if the context is not helpful.                                      If you don't know the answer, say you don't know, but try to provide some helpful information if possible.                         "},
 {'role': 'user',
  'content': 'What is the process for reporting suspected research misconduct at UC?'}]

system_prompt_instances

In [ ]:
import os
import sys

sys.path.append(f"{os.path.dirname(os.path.abspath(__file__))}/../../..")

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from src.maps import LLM_INSTANCE_DICT, LLM_MODEL_DICT
from src.services.llms.prompts import system_prompt as system_prompt_instances
from src.services.llms.prompts import user_prompt as user_prompt_instances
from src.services.llms import tokenizer as tokenizer_instances
from src.services.base import ServiceBase
from utils.module import get_instance

class LLMBase(ServiceBase):
    def __init__(
        self,
        llm_name: str,
        user_prompt_instance_name: str="",
        system_prompt_instance_name: str="",
        service: str="",
        use_example: bool=True,
        seed: int=0,
        is_truncate_response: bool=True,
        is_quantized: bool=False,
        device: str="cuda",
        **kwargs
    ):
        """LLM Base Class as a Service. Check the names from src/maps.py

        Args:
            llm_name (str): LLM base model name.
            user_prompt_instance_name (str, optional): user prompt instance name. Defaults to "".
            system_prompt_instance_name (str, optional): system prompt instance name. Defaults to "".
            use_example (bool, optional): use an example in system prompt. Defaults to True.
            seed (int, optional): seed for response generation. Defaults to 0.
            is_truncate_response (bool, optional): truncate the raw response. Defaults to True.
            is_quantized (bool, optional): whether use quantized model. Defaults to False.
            device (str, optional): use cuda or cpu for LLM. Defaults to "cuda".
            service to feed to GPT system prompt (str, optional): the service for which to set the prompter. Defaults to "".
        """
        super().__init__(**kwargs)

        # Set config.
        # current_dir = os.path.dirname(os.path.abspath(__file__))
        # self.root = f"{current_dir}/../../.."
        self.llm_name = llm_name
        self.use_example = use_example
        self.is_truncate_response = is_truncate_response
        self.seed = seed
        self.is_quantized = is_quantized
        self.device = device

        # Use user prompt for general questions if not specified.
        if user_prompt_instance_name == "":
            user_prompt_instance_name = "GeneralUserPrompt"

        # Build user prompter.
        self.set_user_prompter_by_instance_name(user_prompt_instance_name)

        # Get LLM instance name.
        if llm_name in LLM_INSTANCE_DICT.keys():
            llm_instance_name = LLM_INSTANCE_DICT[llm_name]
        elif llm_name.find("ollama") > -1:
            llm_instance_name = "Ollama"
        ## CK: It seems this is the defult case even when using Ollama integration, 
        # since the model is still GPT-based. 
        # We can further specify the LLM type when we have more LLM types integrated.
        else:
            llm_instance_name = "GPT"

        # Use system prompt by LLM type.
        if system_prompt_instance_name == "":
            system_prompt_instance_name = llm_instance_name

        # Build system prompter.
        self.set_system_prompter_by_instance_name(
            system_prompt_instance_name,
            use_example=use_example,
            service=service
        )

        # Build tokenizer by LLM type.
        self.tokenizer = get_instance(
            tokenizer_instances,
            llm_instance_name
        )(llm_name=llm_name, device=self.device)

        # CPU model cannot support quantization.
        if self.device.find("cpu") > -1:
            is_quantized = False

        # Set quantization configs.
        if is_quantized:
            self.quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
        else:
            self.quantization_config = None

        # Set attention_mask.
        self.attention_mask = lambda system_prompt: \
            torch.any(torch.stack([system_prompt==v for v in [0,1,2]], dim=-1), dim=-1).logical_not()

        # Model and LLM are set from the children class by LLM type.
        self.model = None
        self.llm = None

    def set_user_prompter_by_instance_name(
        self,
        user_prompt_instance_name: str,
        **kwargs
    ):
        """Change user prompter by instance name externally.

        Args:
            user_prompt_instance_name (str): set an user prompter instance name.
        """
        self.user_prompter = get_instance(
            user_prompt_instances,
            user_prompt_instance_name
        )(**kwargs)

    def set_user_prompter(
        self,
        user_prompt_instance: user_prompt_instances.UserPromptBase,
    ):
        """Change user prompter externally.

        Args:
            user_prompt_instance (UserPromptBase): set an user prompter instance.
        """
        self.user_prompter = user_prompt_instance

    def set_system_prompter_by_instance_name(
        self,
        system_prompt_instance_name: str,
        service: str="",
        **kwargs
    ):
        """Change system prompter by instance name externally.

        Args:
            system_prompt_instance_name (str): set a system prompter instance name.
            service (str): the service for which to set the prompter.
        """
        self.system_prompter = get_instance(
            system_prompt_instances,
            system_prompt_instance_name
        )(**kwargs)

    def set_system_prompter(
        self,
        system_prompt_instance: system_prompt_instances.SystemPromptBase,
    ):
        """Change system prompter externally.

        Args:
            system_prompt_instance (SystemPromptBase): set a system prompter instance.
        """
        self.system_prompter = system_prompt_instance

### Cmmented out since it is the same as the one above,
    # def set_system_prompter(
    #     self,
    #     system_prompt_instance: system_prompt_instances.SystemPromptBase,
    # ):
    #     self.system_prompter = system_prompt_instance

    def set_seed(
        self,
        seed: int
    ):
        """Set generation seed externally.

        Args:
            seed (int): generation seed before calling LLM model.
        """
        self.seed = seed

    def set_torch_seed(
        self,
        seed: int
    ):
        """Set PyTorch seed externally.

        Args:
            seed (int): seed for PyTorch program, CPU and GPU.
        """
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)

    def set_truncate_response(
        self,
        is_truncate_response: bool
    ):
        """Set the flag of truncating response externally.

        Args:
            is_truncate_response (bool): truncate the response using key words in system prompt.
        """
        self.is_truncate_response = is_truncate_response

    def truncate_response(
        self,
        response: str
    ):
        """Truncate response.

        Args:
            response (str): raw response from LLM.

        Returns:
            response (str): truncated response.
        """
        # Remove DeepSeek model reasoning part in the response.
        if self.llm_name.find("deepseek") > -1:
            response = response.split("</think>")[-1]

        return response

    def __call__(
        self,
        question: str,
        context: dict = {}
    ):
        """Process the question answering.

        Args:
            question (str): user question in string.
            context (str, optional): context retrieved externally if applicable. Defaults to "".

        Returns:
            response: truncated response.
            raw_response: original response without truncation.
        """
        # Set a seed for reproduction.
        self.set_torch_seed(self.seed)

        # Generate user prompt with question and context.
        user_prompt = self.user_prompter(question, context=context)

        # Merge user prompt to system prompt by LLM type.
        system_prompt = self.system_prompter(user_prompt, context=context)

        # Encode system prompt for LLM.
        system_prompt_encoded = self.tokenizer.encode(system_prompt)

        # Get response from LLM.
        response_encoded = self.llm(system_prompt_encoded)

        # Decode response since some are torch.tensor.
        raw_response = self.tokenizer.decode(response_encoded)

        # Truncate response, is_truncate_response can be set externally by LLM type.
        response = self.truncate_response(raw_response)

        # Return response with and without truncation.
        return response, raw_response

NameError: name '__file__' is not defined

In [63]:
llm = LLMBase(llm_name="qwen2.5:7b", system_prompt_instance_name="GPT", service="")


NameError: name 'tokenizer_instances' is not defined

In [60]:
!pip install torch==2.3.0


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.7 MB/s eta 0:00:0000:0100:01
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (99 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 19.9 MB/s eta 0:00:0000:0100:01
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823

In [37]:
system_prompter = get_instance(
            system_prompt_instances,
            "GPT"
)

system_prompter 


src.services.llms.prompts.system_prompt.GPT